# Data Processing
We started with two datasets. One is a  AI job market dataset with binary skill flags and a real-world job listings dataset with free text skills  and unified them into a single schema with four columns: job_title, skills, experience, and salary. The job_listings dataset was merged to add data with real-world job postings, increasing the dataset from 10,345 to 11,551 rows and introducing a wider, more skills that better reflects the actual job market.

The binary skill flags in the original dataset were collapsed into comma-separated text labels because TF-IDF operates on text tokens, requiring skills to be represented as words rather than numeric 0 or 1 values. This allows the TF-IDF vectorizer to treat each skill as a term across the job listings, computing how frequently a skill appears in a given job and how much it differentiates that job from others. Finally, rows with missing values were dropped to ensure the vectorizer receives clean, complete documents with no null entries that could disrupt tokenization or frequency calculations.


In [9]:
import csv
import pandas as pd

In [7]:
input_f = "AI Job Market Dataset.csv"
output_f = "AI_Job_Market_Transformed.csv"

# The original data is input as binary numbers for skills rather than text. In order to perform IDF,
# we want to process the skills as tokens for TF-IDF

mapping_skills = {
    "skills_python": "python",
    "skills_sql": "sql",
    "skills_ml": "ML",
    "skills_deep_learning": "deep learning",
    "skills_cloud": "cloud",
}

with open(input_f, newline="", encoding="utf-8") as infile:
    reader = csv.DictReader(infile)

    r = []
    #checks if that row's value for that column is "1"
    #  If yes, the label gets added to the list and if not, skip it
    for row in reader:
        skills = [
            label
            for col, label in mapping_skills.items()
            if row.get(col, "0").strip() == "1"
        ]

        r.append({
            "job_title": row["job_title"],
            "skills": ", ".join(skills) if skills else "",
            "experience": row["experience_level"],
            "salary": row["salary"],
        })
# write out the file
with open(output_f, "w", newline="", encoding="utf-8") as outfile:
    fieldnames = ["job_title", "skills", "experience", "salary"]
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(r)

#print test
print(f"finished! {len(r)} r written to {output_f}")

finished! 10345 r written to AI_Job_Market_Transformed.csv


In [8]:
transformed_f = "AI_Job_Market_Transformed.csv"
listings_df = "job_listings.csv"
output_f = "AI_Job_Market_Merged.csv"

r = []

# read the transformed AI job market data
with open(transformed_f, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        r.append({
            "job_title": row["job_title"],
            "skills":    row["skills"],
            "experience": row["experience"],
            "salary":    row["salary"],
        })

# read job_listings and map columns to match
with open(listings_df, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        r.append({
            "job_title":  row["title"],
            "skills":     row["skills"],
            "experience": row["seniority"],
            "salary":     row["salary_estimate_median"],
        })

# write merged output
with open(output_f, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["job_title", "skills", "experience", "salary"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(r)



#print tessts
print(f"done! {len(r)} total r written to {output_f}")
print(f"  - AI Job Market r: {len(r) - sum(1 for r in r if r['experience'] in ['senior','mid','junior','intern','director','principal'])}")


done! 11551 total r written to AI_Job_Market_Merged.csv
  - AI Job Market r: 10394


In [11]:
df = pd.read_csv("AI_Job_Market_Merged.csv")

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)